(C) Copyright 2025 - ECMWF and individual contributors.

This software is licensed under the terms of the Apache Licence Version 2.0
which can be obtained at http://www.apache.org/licenses/LICENSE-2.0.
In applying this licence, ECMWF does not waive the privileges and immunities
granted to it by virtue of its status as an intergovernmental organisation nor
does it submit to any jurisdiction.

# 💿 PoF Data Generator

Importing all the necessary python libraries
before we begin, we must prepare our enviroment. This includes installing the Application Programming Interface (API) of the CDS, and importing the various python libraries we will need.

These should have been installed into a conda environment before running the Notebook.

In [1]:
!pip install -q cdsapi

In [2]:
import cdsapi
import os

from pof_utils import (
    build_training_samples,
    retrieve_ecbox_file,
    retrieve_era5_land_month,
    retrieve_xds_month_to_grib,
)


Here we will define some configuration options. 
For the purpose of the book we try to keep the volume of data small. This will involve downloading data from the CDS.

In [3]:
years = [2003]           # extend this list if needed
months = [1]             # or list(range(1, 13)) for all months


ECBox Data Download

Some of the data we will use is available on an EC Box at ECMWF.
We retrieve the Active Fire maps for the configured months, plus the static population and road density fields. All of these are GRIB2 on the ERA5-Land 0.1° grid, which is the grid every other input is put on. The population file is also used as the template when we write our own GRIB2 files.

In [4]:
os.makedirs("data/CLIMATE", exist_ok=True)

for year in years:
    for month in months:
        retrieve_ecbox_file(f"ACTIVE_FIRE_MAP_{year}_{month:02d}.grib2")

retrieve_ecbox_file("CLIMATE/POP_2020.grib")
retrieve_ecbox_file("CLIMATE/road_density_2015_agg_r.grib")

Failed to retrieve file: https://sites.ecmwf.int/ecbox/POF_DATA/s/dav/data/ACTIVE_FIRE_MAP_2003_01.grib2
404 Not Found


PosixPath('data/CLIMATE/road_density_2015_agg_r.grib')

##Enter your CDS API key and XDS-Preprod API Key

We will request data from the Climate Data Store (CDS) and the Cross Data Store (XDS) programmatically with the help of the CDS API.

If you have not set up your CDS API credentials with a ~/.cdsapirc file, it is possible to provide the credentials when initialising the cdsapi.Client. To do this we must define the two variables needed to establish a connection: URL and KEY . The URL for the cds api is common and you do not need to modify that field. The KEY is string of characters made up of your your personal User ID and CDS API key. To obtain these, first register or login to the CDS (https://cds.climate.copernicus.eu), then visit https://cds.climate.copernicus.eu/how-to-api and copy the string of characters listed after "key:". Replace the ######### below with this string.

For XDS the URL and instance are different as so we need the a different set of credentials for XDS. The API functions in the same way, but you will need to register and login on the XDS (https://xds-preprod.ecmwf.int/) and retrieve your API key here: https://xds-preprod.ecmwf.int/how-to-api 

NOTE: If you have set up your cdsapi key using a ~/.cdsapirc you will need to update it for using the xds key or override it as we have demonstrated below.

In [5]:

cds_kwargs = {
    'url': 'https://cds.climate.copernicus.eu/api',
    'key': None
}

xds_kwargs = {
    'url': 'https://xds.ecmwf.int/api',
    'key': None
}

CDS Data Retrieval

This code block will retrieve the datasets needed from the CDS including temperature, wind speed, precipitation and dewpoint as GRIB. We retrieve every hour of each day for the years and months configured above, and calculate a daily mean for each variable.

ERA5-Land precipitation is accumulated from 00 UTC, so the field at 00 UTC on the next day is the total for the day. We divide it by 24 to give the mean hourly amount. For the last day of the month this needs one extra field from the 1st of the next month.

We require wind speed and relative humidity, which are calculated from the 10U/10V wind components and the 2m temperature/dewpoint for each hour before taking the daily mean. The daily means of temperature, precipitation, relative humidity and wind speed are written to disk as GRIB2, and the hourly files are removed afterwards to save disk space.

In [ ]:
client = cdsapi.Client(**cds_kwargs)
for year in years:
    for month in months:
        retrieve_era5_land_month(client, year, month)

2026-11-11 10:05:48,704 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


Retrieving hourly T2M for 2003-01


2026-11-11 10:05:48,928 INFO Request ID is 3d045bcc-b3f4-44af-bd7b-d0b4b87f769d
2026-11-11 10:05:48,983 INFO status has been updated to accepted
2026-11-11 10:08:42,329 INFO status has been updated to successful


Retrieving hourly D2M for 2003-01


2026-11-11 10:15:21,834 INFO Request ID is d1d68b9a-0cd0-45cb-8809-71700f121cfa
2026-11-11 10:15:23,091 INFO status has been updated to accepted
Recovering from connection error [('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))], attempt 1 of 500
Retrying in 120 seconds
2026-11-11 10:24:21,075 INFO status has been updated to successful


Retrieving hourly 10U for 2003-01


2026-11-11 10:29:09,202 INFO Request ID is 1bb4e3f8-8498-423f-9357-b6119479d268
2026-11-11 10:29:09,271 INFO status has been updated to accepted
2026-11-11 10:31:04,241 INFO status has been updated to running
2026-11-11 10:35:29,482 INFO status has been updated to successful
e0c381cccbf20d3a71e5b747b7335857.grib:  71%|███████   | 2.58G/3.63G [07:24<07:45, 2.43MB/s]

We will now retrieve the dead and live fuel moisture content and the fuel load from the XDS.

The XDS data is delivered as NetCDF on a different grid to ERA5-Land, so once it is downloaded we regrid it onto the ERA5-Land grid and store it as GRIB2.

In [ ]:
client = cdsapi.Client(**xds_kwargs)
for year in years:
    for month in months:
        retrieve_xds_month_to_grib(client, year, month)

Build an XGBoost-compatible training dataset by combining
climate, vegetation, and anthropogenic data from the GRIB files.

Every field is checked to be on the same grid before it is flattened, and only pixels with some fuel are kept.

In [ ]:
output_path = "./data/training_data.parquet"

dfa = build_training_samples(years, months, sample_frac=1)  # keep all data for training

print(f"Saving dataset → {output_path}")
dfa.to_parquet(output_path)
print("✅ Training dataset ready.")